# Projet Final - Traitement des images avec CNN :
### BOU SERHAL Elio
### Master M2 Data Science
### Vendredi 19/12/2024


### Introduction:
1) Objectif du TP : \
L'objectif de ce TP est de mettre en œuvre un modèle de réseau de neurones convolutionnel (CNN) pour la détection d'une rupture de rocher sur des images. \
Nous allons entraîner ce modèle sur un dataset de 40 000 images, où chaque image est associée à une étiquette indiquant la présence ou l'absence d'une rupture. \
Les données seront divisées en batchs de 64 images pour faciliter l'entraînement et la gestion de la mémoire.\

2) Contexte théorique : \
Dans ce TP, nous explorons l'usage des CNN, qui sont des architectures particulièrement efficaces pour la reconnaissance d'images. \
Nous utiliserons ces modèles pour extraire des caractéristiques pertinentes des images et classifier celles-ci en fonction de la présence d'une rupture de rocher. \
Les CNNs sont capables de capturer les relations spatiales au sein des images, ce qui les rend idéals pour cette tâche de classification d'images.


3) Methodologie : \

--> ETAPE 1: Telecharger, Creer load-data et Partisionner la Base de donnee\
Etape 1-1: telecharger les librairies necessaires \
Etape 1-2: Telecharger la base de donnee\
Etape 1-3: Lister, afficher et verifier l'existence des images\
Etape 1-4: Partitionner la base de donnee \

--> ETAPE 2:  Prétraitement des images\
Etape 2-1 : Appliquer des transformations sur les images \
Etape 2-2 : Creation du Data-loaders avec Batch\

--> ETAPE 3: Création du modele CNN \
Etape 3-1: Couche convolutionnelles\
Etape 3-2: Parametres des convolutions\

--> ETAPE 4: Evaluation du modèle

In [1]:
##### ETAPE 1: Telecharger, Creer load-data et Partisionner la Base de donnee ########
# Etape 1-1: telecharger les librairies necessaires 
# Etape 1-2: Telecharger la base de donnee
# Etape 1-3: Lister, afficher et verifier l'existence des images
# Etape 1-4: Partitionner la base de donnee 




# Etape 1-1: telecharger les librairies necessaires 
import kagglehub
import os
from PIL import Image
import glob
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm 




# Etape 1-2: Telecharger la base de donneen
path = kagglehub.dataset_download("arunrk7/surface-crack-detection")
print("Path to dataset files:", path)






# Etape 1-3: Lister, afficher et verifier l'existence des images
# Lister les fichiers
for root, dirs, files in os.walk(path):
    print("Root:", root)
    for file in files:
        print("File:", file)

extensions = ["*.jpg", "*.jpeg", "*.png"]
images = []
for ext in extensions:
    images.extend(glob.glob(os.path.join(path, "**", ext), recursive=True))

# Vérifier si des images ont été trouvées
if not images:
    print("Aucune image trouvée.")
else:
    print(f"{len(images)} images trouvées. Affichage des 5 premières...")

# Affichae des 5 premières images
for img_path in images[:5]:
    try:
        img = Image.open(img_path)
        img.show()  
        print(f"Image affichée : {img_path}")
    except Exception as e:
        print(f"Erreur avec l'image {img_path}: {e}")

Path to dataset files: C:\Users\User\.cache\kagglehub\datasets\arunrk7\surface-crack-detection\versions\1
Root: C:\Users\User\.cache\kagglehub\datasets\arunrk7\surface-crack-detection\versions\1
Root: C:\Users\User\.cache\kagglehub\datasets\arunrk7\surface-crack-detection\versions\1\Negative
File: 00001.jpg
File: 00002.jpg
File: 00003.jpg
File: 00004.jpg
File: 00005.jpg
File: 00006.jpg
File: 00007.jpg
File: 00008.jpg
File: 00009.jpg
File: 00010.jpg
File: 00011.jpg
File: 00012.jpg
File: 00013.jpg
File: 00014.jpg
File: 00015.jpg
File: 00016.jpg
File: 00017.jpg
File: 00018.jpg
File: 00019.jpg
File: 00020.jpg
File: 00021.jpg
File: 00022.jpg
File: 00023.jpg
File: 00024.jpg
File: 00025.jpg
File: 00026.jpg
File: 00027.jpg
File: 00028.jpg
File: 00029.jpg
File: 00030.jpg
File: 00031.jpg
File: 00032.jpg
File: 00033.jpg
File: 00034.jpg
File: 00035.jpg
File: 00036.jpg
File: 00037.jpg
File: 00038.jpg
File: 00039.jpg
File: 00040.jpg
File: 00041.jpg
File: 00042.jpg
File: 00043.jpg
File: 00044.jpg
Fil

Nous passons ensuite à l'étape 1-4 : On divise les données en 2 groupes : \
70 % pour l'entraînement et 30% pour le reste (avec 20% de validation et 10% de test). \
Cette partition nous permette: \
 --> Ajuster le modele sur l'ensemble d'Entrainement \
 --> Evaluer sa performance sur l'ensemble de Validation\
 --> Mesurer sa performance finale sur l'ensemble de Test\
On effectue une boucle sur les ensembles afin que pour chaque ensemble, on traite toutes les images d'une classe donnée. \


On divise en lot afin d'éviter une surcharge de mémoire (le package tqdm nous permet d'afficher une barre de progression), 
les images seront copiées en lot de 500 pour limiter l'utilisation de la mémoire. Chaque image du lot sera copié dans le dossier approprié

In [2]:

# Etape 1-4: Partitionner la base de donnee 

# Il faut avant créer des répertoires pour les ensembles en définissant les chemins où les images seront copiées.
# Chemin pour les ensembles de données 
train_path = "dataset/train"
val_path = "dataset/val"
test_path = "test_path"

# Création des répertoires
os.makedirs(train_path, exist_ok= True)
os.makedirs(val_path, exist_ok= True)
os.makedirs(test_path, exist_ok= True)

# Ensuite identifier les classes dans les sous-dossiers en listant tous les fichiers etr sous-dossiers du chemin.
# Idenfication des classes dans les sous-dossiers
classes = os.listdir(path)
for class_name in classes:
    class_dir = os.path.join(path, class_name)
    if not os.path.isdir(class_dir):
        continue
    images = [os.path.join(class_dir, img) for img in os.listdir(class_dir) if img.endswith(('.jpg', '.png', '.jpeg'))]
    
    # Diviser en train/val/test (70%/20%/10%)
    train_imgs, temp_imgs = train_test_split(images, test_size=0.3, random_state=42)
    val_imgs, test_imgs = train_test_split(temp_imgs, test_size=0.33, random_state=42)  # 0.33 de 30% ≈ 10%

    # Copier les images dans les répertoires correspondants en lots
    for split_name, split_imgs, split_path in [("train", train_imgs, train_path),
                                               ("val", val_imgs, val_path),
                                               ("test", test_imgs, test_path)]:
        print(f"Traitement de la classe {class_name} pour {split_name}...")
        for i in tqdm(range(0, len(split_imgs), 500)):  # Diviser en lots de 500
            batch = split_imgs[i:i + 500]
            dest_dir = os.path.join(split_path, class_name)
            os.makedirs(dest_dir, exist_ok=True)
            for img in batch:
                shutil.copy(img, dest_dir)

print("Les ensembles de données ont été divisés avec succès.")
    
    

Traitement de la classe Negative pour train...


100%|██████████| 28/28 [01:00<00:00,  2.18s/it]


Traitement de la classe Negative pour val...


100%|██████████| 9/9 [00:12<00:00,  1.40s/it]


Traitement de la classe Negative pour test...


...

Traitement de la classe Positive pour val...


100%|██████████| 9/9 [00:12<00:00,  1.35s/it]


Traitement de la classe Positive pour test...


100%|██████████| 4/4 [00:05<00:00,  1.41s/it]

Les ensembles de données ont été divisés avec succès.




Maintenant, preparons les images au modele au CNN. \
On commence par redimensionner, retourner et renverser les images ce qui permet de réduire la complexité, les besoins en mémoire du modèle, rendre les données plus robustes, diverses.
Enfin, entraîne le modèle à être invariant à l'orientation de la fissure (+ de diversification dans les images).\


Ensuite, nous allons ensuite convertir l'image en tenseur en utilisant la valeur de son pixel tel que Chaque pixel sera normalisé,\
Cela permet de préparer l'image à être utilisé par PyTorch.\

Apres, on normalise la valeur des pixels entre [-1, 1] pour augmenter la stabilité numérique et éviter les gradients trop petits ou trop grands.


In [3]:
##### ETAAPE 2:  Prétraitement des images
# Etape 2-1 : Appliquer des transformations sur les images 
# Etape 2-2 : Creation du Data-loaders avec Batch


# Package 
from torchvision import datasets, transforms
from torch.utils.data import DataLoader


# Etape 2-1 : Appliquer des transformations sur les images 
# Définir les transformations
transform_train = transforms.Compose([
    transforms.Resize((128, 128)), # redimension des images en 128x128
    transforms.RandomHorizontalFlip(), # renversement horizontal aléatoire
    transforms.RandomRotation(15), # rotation aléatoiré de 15 degrès
    transforms.ToTensor(), #  convertit l'image en tenseur (valeurs entre [0,1])
    transforms.Normalize([0.5], [0.5]) # normalisation des pixels entre [-1, 1]
])

transform_test = transforms.Compose([
    transforms.Resize((128, 128)),          
    transforms.ToTensor(),                  
    transforms.Normalize([0.5], [0.5])     
])

# Chargements des images 
train_dataset = datasets.ImageFolder(root=train_path, transform=transform_train)
val_dataset = datasets.ImageFolder(root=val_path, transform=transform_test)
test_dataset = datasets.ImageFolder(root=test_path, transform=transform_test)







# Etape 2-2 : Creation du Data-loaders avec Batch
# Création DataLoaders avec batchs
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

print("Données prétraitées et organisées en batchs pour l'entraînement.")

Données prétraitées et organisées en batchs pour l'entraînement.


Maintenant, on doit creer notre modele de classification :\
Le modele CNN (Convolution Neural Network) est utilise pour traiter des données ayant une structure spatiale (comme les images de fissures ou non dans notre cas). \
Il est composé de plusieurs types de couches : \
--> les couches convolutionnelles (a) qui extraient les caractéristiques locales des images et capturent des motifs, \
--> les couches de pooling (b) qui réduisent la dimension des caractéristiques et permettent de diminuer le nombre de paramètres à apprendre \
--> les couches denses (c) qui effectuent la classification. \


**(a) Couche convolutionnelle :**\
y[m, n] = sum_{u=-k}^k \ sum_{v=-k}^k x[m+u, n+v] \ cdot w[u, v] + b \
nn.Conv2d(): applique une convolution sur l'image  d'entrée elle prend en paramètres le nombre de canaux (ici 3 pour RGB), \
le nombre de filtres et sa taille, le déplacement du filtre sur chaque pixel et la bordure pour conserver sa taille.\
Nous appliquons ensuite une fonction d'activation ReLU afin d'introduire la non-linéarité. \
Cela va remplacer toutes les valeurs négatives par 0, accélèrer la convergence et permettre de réduire les problèmes de saturation: f(x) = max(0, x) \



**(b) Couche de Pooling :**\
Pour effecturer un sous-échantillonage en prenant le maximum dans les régions locales. On réduit ainsi la dimension en gardant les caractéristiques les plus importantes.\
y[m, n] = \max \{x[i, j] : i \in [m, m+s), j \in [n, n+s)\} \
La couche dense nous permet de sortir une probabilité de taille 1 : y = W \cdot x + b \
Et la fonction sigmoid permet la classification binaire afin de produire une probabilité entre 0 et 1 : f(x) = \frac{1}{1 + e^{-x}} \




In [ ]:
##### ETAPE 3: Création du modele CNN 
# Etape 3-1: Couche convolutionnelles
# Etape 3-2: Parametres des convolutions


# Packages 
import torch
import torch.nn as nn
import torch.optim as optim

class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        # Correction de la dimension d'entrée dans nn.Linear
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 32 * 32, 128),  # Utilisation de la dimension correcte 65536
            nn.ReLU(),
            nn.Linear(128, 1),  # 1 sortie pour la classification binaire
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.conv_layers(x)  # Passe dans les convolutions
        x = self.fc_layers(x)    # Passe dans les couches entièrement connectées
        return x

    # Méthode pour tester les dimensions
    def test_dimensions(self, input_size):
        x = torch.randn(*input_size)  # Crée un tenseur factice
        print(f"Entrée : {x.shape}")
        for layer in self.conv_layers:
            x = layer(x)
            print(f"Après {layer}: {x.shape}")
        x = torch.flatten(x, start_dim=1)
        print(f"Après aplatissement : {x.shape}")

model = SimpleCNN()
model.test_dimensions((1, 3, 128, 128))  # Batch size = 1, canaux = 3, taille = 128x128

Entrée : torch.Size([1, 3, 128, 128])
Après Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)): torch.Size([1, 32, 128, 128])
Après ReLU(): torch.Size([1, 32, 128, 128])
Après MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False): torch.Size([1, 32, 64, 64])
Après Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)): torch.Size([1, 64, 64, 64])
Après ReLU(): torch.Size([1, 64, 64, 64])
Après MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False): torch.Size([1, 64, 32, 32])
Après aplatissement : torch.Size([1, 65536])


Nous avons d'abord commencer par tester la dimension des données afin de nous assurer que chaque étape du réseau est compatible avec les couches suivantes. 
Nous avons tester ces dimensions avec un tenseur factice afin d'observer les dimensions exactes après chaque couche convolutionnelle et pooling, 
et la taille exacte du vecteur applati utilisé dans la couche dense. 
De plus, l'utilisation de tenseurs factices pour tester le flux de données est une bonne pratique qui revient fréquemment dans la littérature.


Justification de notre architecture pour SimpleCNN:\
 **Couches convolutionnelle**:\
Chaque couche convolutionnelle détecte des motifs locaux spécifiques. En empilant les couches, on commence par détecter les motifs de bas niveau \
puis avec la deuxième couche on détecte les caractéristiques plus complexes comme les formes ou textures spécifiques.\
Dans notre cas, la première couche détectera les contours de la fissure tandis que la deuxième convolution combinera les contours afin de former une représentation plus abstraite. \
La première a 32 filtres afin de capturer une variété de motifs, et la deuxième de 64 filtres capture les caractéristiques plus complexes 
en construisant sur les motifs détectés par la première couche.\

**Paramètres des convolutions** : \
- La taille de kernel est classique, la littérature en lien nous indique qu'il est courant de conserver la taille du noyau à 3x3 afin de capter les motifs en restant efficace.\
- Le padding, qui contrôle la taille de la sortie après convolution a été fixé à 1 ce qui signifie qu'une bordure de 1 pixel est ajoutée autour de l'image d'entrée. \
    Cela nous permet de ne pas trop écraser les dimensions de l'image, ce qui est utile pour éviter la réduction trop rapide de la taille, surtout dans les architectures profondes comme dans notre cas. 
- Le pas de déplacement a été fixé à 1, il permet de contrôler de combien de pixels le filtre se déplace après chaque opération. \
    Un stride plus élevé parcourerait l'image trop rapidement sans sans se concentrer sur les motifs locaux. \
- Activation ReLU() afin d'introduire une non-linéarité pour apprendre des représentations complexes et réduire le risque de saturation des gradients. \
- Chaque convolution est suivie d'un pooling afin de réduire les dimensions spatiales en limitant le nombre de paramètres pour conserver les caractéristiques importantes.\

**Couches entièrement connectées** :\
- Les sorties convolutives sont aplaties en un vecteur pour les rendre compatibles avec les couches entièrement connectées.\
- Une réduction à 128 neurones crée une couche intermédiaire capable de capturer les relations entres les caractéristiques extraites par les convolutions.\
- Une seule couche de sortie pour la probabilité de la classe "fissure" car le problème est binaire et ne nécessite qu'un score unique, \
    le modèle produit une seule valeur afin de produire une probabilité entre [0,1]. \
        Le soft max est surtout utilisé pour les problèmes multi-classe, c'est pourquoi nous ne l'utilisons pas ici.


Maintenant il faut configurer la fonction de perte et l'optimiseur avant de passer à l'entraînement du modèle. \
La fonction de perte mesure l'écart entre les prédictions du modèle et les vraies valeurs. \
Elle guide l'entraînement du modèle en indiquant dans quelle direction ajuster les poids pour réduire l'écart.\
Dans notre cas, la fonction Binarry Cross-Entropy Loss est la plus adaptée car elle est spécifiquement conçue pour les classifications binaires de notre projet. \
La fonction de perte est définie comme suit : \
mathcal{L}(y, \hat{y}) = - \big( y \cdot \log(\hat{y}) + (1 - y) \cdot \log(1 - \hat{y}) \big) \

Si la probabilité est proche de la vraie étiquette, la perte est faible. Et pour un batch de taille $N$, on a : \
mathcal{L}_{\text{moyenne}} = \frac{1}{N} \sum_{i=1}^N \mathcal{L}(y_i, \hat{y}_i) 

In [5]:
import torch.nn as nn
import torch.optim as optim

# initialisation modèle
model = SimpleCNN()

# BCE
criterion = nn.BCELoss()

# Optimiseur 
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr = learning_rate)

print("Fonction de perte et optimiseur configurés.")

Fonction de perte et optimiseur configurés.


Tout d'abord, toutes les couches du CNN sont initialisées avec poids et biais aléaoires (par défaut de PyTorch). \
On lance ensuite la fonction de perte, et on définit l'optimiseur **Adam** : \
un optimiseur qui permet d'adapter automatiquement le taux d'apprentissage pour chaque paramètre en fonction de l'historique des gradients, \
il est robuste et plus adapté dans le cas d'architecture CNN. \

**Optimiseur Adam** : \
theta_t = \theta_{t-1} - \alpha \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
avec correction des biais : \
hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t} \
On choisit 0.001 de taux d'apprentisage pour qu'il ne soit ni trop grand ou petit. \

Notre objectif ici est de minimiser la fonction de perte qui mesure l'écart entre les prédictions du modèle et les étiquettes réelles.\
La fonction de perte, implémentée plus haut, est BCE : mathcal{L}(y, \hat{y}) = -\frac{1}{N} \sum_{i=1}^N \Big( y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i) \Big) \

Pour chaque image, le modèle va extraire les caractéristiques locales de l'image avec les couches convolutionnelles et combiner les caractéristiques \
avec les couches entièrement connectées afin d'effectuer la classification. On aura alors une sortie sigmoïde en probabilité : hat{y} = \frac{1}{1 + e^{-z}} \

La passe arrière, qui consiste au calcul des gradients permettra de les calculer en comparant aux poids du modèle : \
frac{\partial \mathcal{L}}{\partial w} = \frac{\partial \mathcal{L}}{\partial \hat{y}} \cdot \frac{\partial \hat{y}}{\partial z} \cdot \frac{\partial z}{\partial w} \

Les poids sont mis à jour selon: \
w_{\text{nouveau}} = w_{\text{ancien}} - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \cdot \hat{m}_t \
m_t = \beta_1 m_{t-1} + (1 - \beta_1) \nabla \mathcal{L} \
v_t = \beta_2 v_{t-1} + (1 - \beta_2) (\nabla \mathcal{L})^2 \

Pendant l'entraînement, on pourra accumuler la perte sur tous les batchs afin de calculer la perte moyenne par époque: \
mathcal{L}_{\text{époque}} = \frac{1}{N_{\text{batchs}}} \sum_{\text{batchs}} \mathcal{L}_{\text{batch}} \

In [6]:
#### ETAPE 3: Entraînement du modèle 
import torch

# Configurer les paramètres
num_epochs = 10  # Nombre d'époques
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # nous ne disponsons pas de GPU, mais commande utile dans ce cas 

# Envoyer le modèle sur le device (si GPU disponible - donc ici CPU)
model = model.to(device)

for epoch in range(num_epochs):
    print(f"Époque {epoch+1}/{num_epochs}")
    model.train()  # Mettre le modèle en mode entraînement

    running_loss = 0.0  # initialisation perte cumulée

    for inputs, labels in train_loader:
        # Envoyer les données sur le device
        inputs, labels = inputs.to(device), labels.to(device)

        # Réinitialiser les gradients
        optimizer.zero_grad()

        # Passe avant
        outputs = model(inputs)  # Prédictions
        loss = criterion(outputs.squeeze(), labels.float())  # Calcul de la perte

        # Passe arrière
        loss.backward()  # Calcul des gradients
        optimizer.step()  # Mise à jour des poids

        # Ajouter la perte courante au total
        running_loss += loss.item()

    # Afficher la perte moyenne pour cette époque
    print(f"Perte moyenne sur l'ensemble d'entraînement : {running_loss / len(train_loader):.4f}")

Époque 1/10
Perte moyenne sur l'ensemble d'entraînement : 0.0662
Époque 2/10
Perte moyenne sur l'ensemble d'entraînement : 0.0287
Époque 3/10
Perte moyenne sur l'ensemble d'entraînement : 0.0205
Époque 4/10
Perte moyenne sur l'ensemble d'entraînement : 0.0135
Époque 5/10
Perte moyenne sur l'ensemble d'entraînement : 0.0155
Époque 6/10
Perte moyenne sur l'ensemble d'entraînement : 0.0220
Époque 7/10
Perte moyenne sur l'ensemble d'entraînement : 0.0108
Époque 8/10
Perte moyenne sur l'ensemble d'entraînement : 0.0080
Époque 9/10
Perte moyenne sur l'ensemble d'entraînement : 0.0071
Époque 10/10
Perte moyenne sur l'ensemble d'entraînement : 0.0125


La pipeline réalisée est adaptable dans des situations où un GPU est disponible.\
Le code ci-dessus passe **10 fois** sur les données d'entraînement. \
Dans le cadre de notre projet, un choix de 10 époques est relativement justifiée d'autant que le modèle n'est pas des plus complexes. \
A chaque époque, le modèle est mis en mode entraînement et une boucle parcourt tous les batchs de données d'entraînement. \
Les données sont envoyées sur le device et les gradients sont remis à zéro. \
Une passe avant est effectué afin de fare des prédictions sur le batch et nous calculons l'écart entre les prédictions et les labels réels (perte). \
Une passe arrière calcule les gradients pour ajuster les poids qui seront mis à jour avec Adam. \
Une fois tous les batchs traités, la perte moyenne pour l'époque est affichée pour l'amélioration du modèle \

**Observations générales sur l'entrainement du modèle** :\
On remarque une diminution constante de la perte assurant ainsi que le modèle converge efficacement. \
Les fluctuations dans les pertes sont aussi moindres. 


In [7]:
#### ETAPE 4: Evaluation du modèle


# Boucle de prédiction
test_loss = 0.0
all_predictions = []
all_labels = []

with torch.no_grad():  # Désactiver le calcul des gradients pour économiser de la mémoire
    for inputs, labels in test_loader:
        # Envoyer les données sur le même device que le modèle
        inputs, labels = inputs.to(device), labels.to(device)
        
        # Prédictions
        outputs = model(inputs)
        predictions = (outputs.squeeze() >= 0.5).float()  # Seuil de 0.5 pour la classification

        # Calcul de la perte pour suivre les performances
        loss = criterion(outputs.squeeze(), labels.float())
        test_loss += loss.item()
        
        # Stocker les prédictions et labels pour calculer les métriques
        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Perte moyenne sur l'ensemble de test
test_loss /= len(test_loader)
print(f"Perte moyenne sur l'ensemble de test : {test_loss:.4f}")

# %% [markdown]
# Nous allons calculer également quelques métriques afin de s'assurer des performances globales du modèle. 

# %%
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, roc_auc_score, roc_curve

# Précision
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Précision : {accuracy:.4f}")

# Matrice de confusion
cm = confusion_matrix(all_labels, all_predictions)
print(f"Matrice de confusion :\n{cm}")

# courbe ROC et AUC
auc = roc_auc_score(all_labels, all_predictions)
print(f"AUC : {auc:.4f}")

Perte moyenne sur l'ensemble de test : 0.0078
Précision : 0.9975
Matrice de confusion :
[[1977    3]
 [   7 1973]]
AUC : 0.9975


L'évaluation va nous permettre de mesurer la perfomance du modèle.\
Nous allons donc parcourir les données de test par batchs, effectuer les prédictions pour chaque batch et enfin les comparer avec les étiquettes réelles.
model.eval() # On passe le modèle en mode évaluation


La perte de moyenne sur l'ensemble de test est de **00.0078**, ce qui est faible et qui indique que notre modèle est bien capable de prédire \
les classes fissure/non-fissure sur les données jamais observées. On observe également une précision de 99.60%, le modèle classe correctement presque tous les échantillons, \
il fait très peu d'erreurs, ce qui est confirmé par la matrice de confusion.\

L'AUC, qui mesure la capacité du modèle à discriminer entre les classes est proche de 1, \
    il y a donc une forte séparation entre fissure/non-fissure.\

Nous rappelons que les données utilisées pour cette évaluation n'ont pas été vues, ce qui garantit une évaluation impartiale. 




**Source** : \
https://inside-machinelearning.com/cnn-couche-de-convolution/ \
https://stanford.edu/~shervine/l/fr/teaching/cs-230/pense-bete-reseaux-neurones-convolutionnels \
https://pytorch.org/docs/stable/torch.compiler_fake_tensor.html \
https://datascientest.com/qu-est-ce-qu-un-epoch-en-machine-learning \